In [ ]:
import time
import os
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session
import rasterio
import matplotlib.pyplot as plt

In [ ]:
# Paste your credentials here when running - delete before sharing
CLIENT_ID = ""
CLIENT_SECRET = ""

# Authenticate
client = BackendApplicationClient(client_id=CLIENT_ID)
oauth = OAuth2Session(client=client)

token = oauth.fetch_token(
    token_url='https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
    client_secret=CLIENT_SECRET,
    include_client_id=True
)

print("Authentication successful:", token is not None)

In [ ]:
os.makedirs('data/sentinel2', exist_ok=True)

# set the time windows
years = range(2016, 2026)
months = {
    '05': 'may', '06': 'jun', '07': 'jul',
    '08': 'aug', '09': 'sep', '10': 'oct', '11': 'nov'
}

#Longitude and Latitude for Butte County
lat_min = 39.15   
lat_max = 40.31   
lon_min = -122.22 
lon_max = -120.93 

# set bounding box to this region
bbox = [lon_min, lat_min, lon_max, lat_max]

for year in years:
    for month_num, month_name in months.items():
        
        filename = f'data/sentinel2/{year}_{month_num}_{month_name}.tif'
        
        # Skip if already downloaded
        if os.path.exists(filename):
            print(f"Skipping {filename} — already exists")
            continue
        
        evalscript = """
        //VERSION=3
        function setup() {
          return {
            input: ["B04", "B08", "B11", "B12", "SCL"],
            output: { bands: 5, sampleType: "FLOAT32" }
          }
        }
        function evaluatePixel(sample) {
          return [sample.B04, sample.B08, sample.B11, sample.B12, sample.SCL]
        }
        """
        
        json_request = {
            "input": {
                "bounds": {
                    "bbox": bbox,
                    "properties": {"crs": "http://www.opengis.net/def/crs/EPSG/0/4326"}
                },
                "data": [{
                    "type": "sentinel-2-l2a",
                    "dataFilter": {
                        "timeRange": {
                            "from": f"{year}-{month_num}-01T00:00:00Z",
                            "to": f"{year}-{month_num}-28T23:59:59Z"
                        },
                        "maxCloudCoverage": 20,
                        "mosaickingOrder": "leastCC"
                    }
                }]
            },
            "output": {
                "resx": 0.0045,
                "resy": 0.0045,
                "responses": [{
                    "identifier": "default",
                    "format": {"type": "image/tiff"}
                }]
            },
            "evalscript": evalscript
        }
        
        url = "https://sh.dataspace.copernicus.eu/process/v1"
        headers = {"Authorization": f"Bearer {token['access_token']}"}
        response = oauth.post(url, headers=headers, json=json_request)
        
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print(f"Saved {filename} ({len(response.content)/1000:.0f} KB)")
        else:
            print(f"Failed {filename} — status {response.status_code}")
        
        # Small pause between requests
        time.sleep(1)

print("Done!")

In [ ]:
## Some months (October 2016 for example) had a very small file size, which we remove from the files 

small_files = []

for f in os.listdir('data/sentinel2'):
    path = f'data/sentinel2/{f}'
    size = os.path.getsize(path)
    if size < 10000:  # less than 10KB
        small_files.append((f, size))

for f, size in small_files:
    os.remove(f'data/sentinel2/{f}')
    print(f"Deleted {f}")